In [1]:
import pandas as pd
import numpy as np

In [2]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

In [4]:
df = pd.read_excel(r"C:\Users\ravik\Desktop\CSTR digital twin\data\combined_data_cleaned.xlsx")

In [5]:
#  collecting some data for verification of the data point 
# for i in range(4900):
#     if df["PDO_fraction"][i]>0.18:
#         break
# print(i)


In [6]:
# test_row = df.iloc[126,0:6][1:]
# test_row = np.array(test_row).reshape(1,-1)
# test_row.shape
# # x_test_row  = np.array(test_row).reshape()

In [7]:
# x_test_row  = np.array(test_row).reshape(5,1)
# x_test_row

In [8]:
X = df.iloc[:,1:6]
X.sample(1)


,Unnamed: 0,R_temp,R_vol,H2_flow,feed1_flow
2435,2535,240.142403,1322.973762,10.892797,145.761243


In [9]:
Y = df["PDO_fraction"]
Y.head(4)

0    0.097191
1    0.119857
2    0.109756
3    0.157642
Name: PDO_fraction, dtype: float64

In [10]:
x_train,x_test,y_train,y_test = train_test_split(X,Y,test_size=0.3)

In [11]:
y_train.shape

(3430,)

In [12]:
rfr = RandomForestRegressor(n_estimators=300)
rfr.fit(x_train,y_train)

,n_estimators,300
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [13]:
y_pred = rfr.predict(x_test)

In [14]:
r2_score(y_test,y_pred)

0.94186099233035

In [15]:
rfr.predict(test_row)

NameError: name 'test_row' is not defined

In [ ]:
xgb_model = xgb.XGBRegressor(n_estimators=200,
                            learning_rate = 0.1
                            )

In [ ]:
xgb_model.fit(x_train,y_train)
y_pred1 = xgb_model.predict(x_test)

In [ ]:
r2_score(y_test,y_pred1)

0.9989399740661694

In [ ]:
from sklearn.model_selection import cross_val_score

## We will use Optuna to optimize hyperparameters

In [ ]:
import optuna

## Optuna on Randomforest

In [ ]:
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    model = RandomForestRegressor(
        n_estimators = n_estimators,
        max_depth = max_depth,
        random_state = 42
        
    )

     # Perform 3-fold cross-validation and calculate r2_score
    score = cross_val_score(model, x_train, y_train, cv=3, scoring='r2').mean()
    return score

        
    

    
    

In [ ]:
study= optuna.create_study(direction='maximize')
study.optimize(objective,n_trials=1)

print("Best Parameters:", study.best_params)
print("Best R²:", study.best_value)

[I 2026-06-12 17:37:25,096] A new study created in memory with name: no-name-ffa5957f-c1c2-4a8f-9df2-f5114b63d88b
[I 2026-06-12 17:37:29,559] Trial 0 finished with value: 0.9957602631067876 and parameters: {'n_estimators': 65, 'max_depth': 20}. Best is trial 0 with value: 0.9957602631067876.
[I 2026-06-12 17:37:34,539] Trial 1 finished with value: 0.9958012346226212 and parameters: {'n_estimators': 74, 'max_depth': 17}. Best is trial 1 with value: 0.9958012346226212.
[I 2026-06-12 17:37:42,546] Trial 2 finished with value: 0.9823645663674712 and parameters: {'n_estimators': 228, 'max_depth': 6}. Best is trial 1 with value: 0.9958012346226212.
[I 2026-06-12 17:37:51,781] Trial 3 finished with value: 0.9940219980542083 and parameters: {'n_estimators': 201, 'max_depth': 9}. Best is trial 1 with value: 0.9958012346226212.
[I 2026-06-12 17:38:02,513] Trial 4 finished with value: 0.9957792818640439 and parameters: {'n_estimators': 179, 'max_depth': 13}. Best is trial 1 with value: 0.99580123

Best Parameters: {'n_estimators': 293, 'max_depth': 16}
Best R²: 0.9958651112383166


In [ ]:

best_rfr_model = RandomForestRegressor(
    n_estimators= 293,
    max_depth=16,
    random_state=42
)

In [ ]:
best_rfr_model.fit(x_train,y_train)

,n_estimators,293
,criterion,'squared_error'
,max_depth,16
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [ ]:
x_test1 = np.array(([2.55469671e+02, 1.16728556e+03, 2.98061003e+01, 8.70518628e+01,
       9.15892103e-01])).reshape(1,5)
x_test1.shape


(1, 5)

In [ ]:
best_rfr_model.predict(x_test1)

C:\Users\ravik\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


array([0.19483191])

In [ ]:
y_pred3 = best_rfr_model.predict(x_test)

In [ ]:
r2_score(y_test,y_pred3)

In [ ]:
print(r2_score(y_test, y_pred3))
print(mean_absolute_error(y_test, y_pred3))
print(mean_squared_error(y_test, y_pred3)**0.5)
print(mean_absolute_percentage_error(y_test, y_pred3))

0.9982149990130071
0.0006853745150640122
0.0012303143321963066
0.006983344494674406


## Optuna on XgboostRegressor

In [ ]:
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 400)
    max_depth = trial.suggest_int('max_depth', 3, 20)
    learning_rate = trial.suggest_float('learning_rate',0.01,1)

    model = xgb.XGBRegressor(
        n_estimators = n_estimators,
        max_depth = max_depth,
        learning_rate = learning_rate,
        random_state = 42
        
    )

     # Perform 3-fold cross-validation and calculate r2_score
    score = cross_val_score(model, x_train, y_train, cv=3, scoring='r2').mean()
    return score


In [ ]:
study = optuna.create_study(direction= "maximize")
study.optimize(objective,n_trials=30)

print("Best Parameters:", study.best_params)
print("Best R²:", study.best_value)

[I 2026-06-12 18:45:00,061] A new study created in memory with name: no-name-2ca4912f-e123-4f8f-bb44-57fb6e1dc968
[I 2026-06-12 18:45:03,930] Trial 0 finished with value: 0.9974757072131862 and parameters: {'n_estimators': 235, 'max_depth': 4, 'learning_rate': 0.0989700941061068}. Best is trial 0 with value: 0.9974757072131862.
[I 2026-06-12 18:45:08,560] Trial 1 finished with value: 0.9928183138526391 and parameters: {'n_estimators': 258, 'max_depth': 3, 'learning_rate': 0.8897086030384987}. Best is trial 0 with value: 0.9974757072131862.
[I 2026-06-12 18:45:12,132] Trial 2 finished with value: 0.9904686160644678 and parameters: {'n_estimators': 389, 'max_depth': 8, 'learning_rate': 0.9518256599705764}. Best is trial 0 with value: 0.9974757072131862.
[I 2026-06-12 18:45:14,526] Trial 3 finished with value: 0.9892862446645695 and parameters: {'n_estimators': 136, 'max_depth': 14, 'learning_rate': 0.9823863131005621}. Best is trial 0 with value: 0.9974757072131862.
[I 2026-06-12 18:45:1

Best Parameters: {'n_estimators': 235, 'max_depth': 4, 'learning_rate': 0.0989700941061068}
Best R²: 0.9974757072131862


In [ ]:
best_xgb_model = xgb.XGBRegressor(n_estimators =235,
                                        max_depth = 4,
                                 learning_rate = 0.0989700941061068)


In [ ]:
best_xgb_model.fit(x_train,y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [ ]:
y_pred4 = best_xgb_model.predict(x_test)
r2_score(y_test,y_pred4)

0.9980741435804824

In [ ]:
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error
)

In [ ]:
print(r2_score(y_test, y_pred4))
print(mean_absolute_error(y_test, y_pred4))
print(mean_squared_error(y_test, y_pred4)**0.5)
print(mean_absolute_percentage_error(y_test, y_pred4))

0.9980741435804824
0.0008031387387124271
0.0012779351205995223
0.008094378955913452


In [ ]:
import pickle

## Based on the R2_score,MAE,RMSE metrics the Random forest is performing good

In [60]:
with open('forward_model_PDO_fraction.pkl','wb') as file:
    pickle.dump(best_rfr_model,file)